# Paper-Ready Metric Plots

PSNR / SSIM vs transmission budget. Error bars = 95% CI of the mean: `mean ± 1.96 × (σ/√n)`.

**Run from `dlapisgs-utility/`:**
```bash
jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=120 \
    plotting/paper_plot_metrics.ipynb
```

In [1]:
import sys
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")

In [2]:
# ------------------------------- setting start ------------------------------ #
# RULE: no plot in this notebook gets a suptitle/figure title. Captions live in the
# LaTeX paper, not the PNG. Per-panel titles inside a multi-scene grid (plot_grid's
# ax.set_title(scene)) are the only exception -- they label panels, not the figure.
# Quick throwaway/debug plots outside this notebook can have titles; paper figures can't.
color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
errorbar_color = "#3A3A3A"

# font
csfont = {'family': 'serif', 'serif': ['Times New Roman', 'Times'], 'size': 23}

# errorbar plot size
err_lw       = 1.5
err_capsize  = 4
err_capthick = 1.5

# figure size
figsize = (6.4, 4.8)

# set theme first, then rc — so seaborn doesn't clobber the font size
sns.set_theme(style="ticks", font="Times New Roman")
plt.rc('text', usetex=True)
plt.rc('font', **csfont)
plt.rcParams['text.latex.preamble'] = r'\usepackage{mathptmx}'
# -------------------------------- setting end ------------------------------- #

In [3]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# 2026-07-13: exp1 is a slice of the comprehensive Exp4 grid sweep (grid8, progressive
# packing, vd_lod scheme, weight_mode as the swept dimension) -- not a separate run. Using
# the same fixed-ml/lpips-enabled CSV as Exp4 instead of the old standalone exp1 CSV, which
# predates the ml label fix and has no LPIPS.
SUMMARY_CSV = "../VCUTS_backup/output/0704/quality_sweep/summary_all_ml_fixed.csv"  # archived 2026-07-17, still the only source with volume/volume_over_d2 weight_mode
OUT_DIR     = "plotting/paper/exp1_weights/"
GROUP_BY    = "weight_mode"
EXCLUDE_KEYS = ["random"]
SLICE_FILTERS = {"grid_shape": "[8, 8, 8]", "packing_mode": "progressive", "scheme": "vd_lod"}

BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

color_palette = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf"]

# canon (2026-07-03): sum-w_mode exp2 rerun showed vd_lod_w (with distance term) craters on
# real scenes -- v_lod_w (W_k only, no distance) is now THE deployed heuristic ("ours").
# vd_lod_w kept in KEY_CONFIG for historical/retired reference, excluded from exp2 plots.
KEY_CONFIG = {
    "screen_area":      {"label": "Proj. Area (Ours)",             "marker": "o", "color": color_palette[4]},
    "volume":           {"label": "Volume",                        "marker": "^", "color": color_palette[2], "ls": (0, (1, 1))},
    "volume_over_d2":   {"label": r"Volume/$d^2$ (Ours)",             "marker": "x", "color": color_palette[3]},
    # "random":           {"label": "Random (control)",              "marker": "x", "color": color_palette[7]},
    "vd_lod":           {"label": "Tile-Based",                       "marker": "s", "color": color_palette[0], "ls": (0, (1, 1))},
    "vd_lod_w":         {"label": "Heuristic (ours, retired)",             "marker": "^", "color": color_palette[1]},
    "v_lod_w":          {"label": "Heuristic (Ours)",         "marker": "v", "color": color_palette[5]},
    "ml":               {"label": "ML (Ours)",                          "marker": "P", "color": color_palette[2]},
    "oracle_loo":       {"label": "Proxy Oracle",       "marker": "*", "color": color_palette[3]},
    "prog_cull":        {"label": "GS (culled)",    "marker": "o", "color": color_palette[0]},
    "prog_no_cull":     {"label": "GS (not culled)",      "marker": "^", "color": color_palette[2]},
    "tile_strict":      {"label": "Tiles",                   "marker": "s", "color": color_palette[6]},
    "tiled_ml":         {"label": "Tiles Learned",        "marker": "P", "color": color_palette[1]},
    "tiled_oracle":     {"label": "Tiles Oracle",      "marker": "*", "color": color_palette[3]},
}

KEY_ORDER = {
    "weight_mode": ["volume", "volume_over_d2", "screen_area", "random"],  # baseline (volume) first
    "scheme":      ["vd_lod", "v_lod_w", "ml", "oracle_loo"],
    "condition":   ["prog_cull", "prog_no_cull", "tiled_ml", "tiled_oracle"],
}

DPI = 300


In [4]:
import os, sys

# cd to dlapisgs-utility/ regardless of where nbconvert was invoked
_cwd = Path(os.getcwd())
_root = None
_candidates = [_cwd] + list(_cwd.parents) + [Path(p) for p in sys.path]
for _candidate in _candidates:
    if (_candidate / "utility_calculation.py").exists():
        _root = _candidate
        break
if _root is None:
    try:
        _root = Path(__file__).resolve().parent.parent
    except NameError:
        _root = _cwd
os.chdir(_root)
print(f"cwd: {Path.cwd()}")

cwd: /mnt/data1/samk/gs-quic/cs5262_tile_quic/dlapisgs-utility


In [5]:
# shared DATA pipeline: single source of truth with experiments/plot_metrics.py
# (rendering is paper-specific and lives in this notebook, like plot_metric below)
sys.path.insert(0, str(Path.cwd()))
from experiments.plot_metrics import (
    _apply_budget_labels, _aggregate, _resolve_order_and_labels,
    _data_ylim, _bk_sort, PSNR_SATURATION_DB,
)

In [6]:
summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)
for _col, _val in SLICE_FILTERS.items():
    df = df[df[_col] == _val]
df = df[~df[GROUP_BY].isin(EXCLUDE_KEYS)].copy()

# provenance: exact sliced rows behind this experiment's figures, alongside the PNGs
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

# convert to list-of-dicts (plot_metrics format) and attach per-scene budget labels
rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")


Loaded 36000 rows | groups: ['screen_area', 'volume', 'volume_over_d2'] | budgets: ['10%', '25%', '40%', '55%', '70%', '85%', '99%', '100%']


In [7]:
agg    = _aggregate(rows, GROUP_BY)
pm_order, pm_labels = _resolve_order_and_labels(agg, GROUP_BY)

# paper ordering: prefer KEY_ORDER override, fall back to plot_metrics order
preferred = KEY_ORDER.get(GROUP_BY, pm_order)
order = [k for k in preferred if k in agg] + [k for k in pm_order if k not in preferred and k in agg]

# labels: KEY_CONFIG overrides plot_metrics defaults
labels = {k: KEY_CONFIG[k]["label"] if k in KEY_CONFIG else pm_labels.get(k, k) for k in order}

# per-scene aggregates for grid plot
from collections import defaultdict as _dd
_by_scene = _dd(list)
for r in rows:
    _by_scene[r["scene"]].append(r)
SCENE_ORDER = ["bicycle", "garden", "stump", "chair", "drums", "ficus", "hotdog", "materials", "mic", "ship"]
scenes_agg = {s: _aggregate(_by_scene[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene}

sample_key = order[0]
sample_bk  = sorted(agg[sample_key], key=_bk_sort)[0]
print(f"agg sample  {GROUP_BY}={sample_key!r}  budget={sample_bk}:")
print(agg[sample_key][sample_bk])
print(f"scenes: {list(scenes_agg.keys())}")

agg sample  weight_mode='volume'  budget=10%:
{'psnr_mean': 22.111437811533612, 'psnr_ci95': 2.15462879944429, 'ssim_mean': 0.8229699765990178, 'ssim_ci95': 0.03960992423644104, 'lpips_mean': 0.22651083872754438, 'lpips_ci95': 0.037494639883133674, 'ngs_mean': 187667.6, 'ngs_ci95': 160051.17141777815, 'n': 10, 'n_cameras': 1500}
scenes: ['bicycle', 'garden', 'stump', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship']


In [8]:
def plot_metric(agg, order, metric, ylabel, out_stem, out_dir, key_config=None):
    key_config = KEY_CONFIG if key_config is None else key_config
    fallback_markers = ["s", "^", "D", "o", "v", "P", "X"]

    fig, ax = plt.subplots(figsize=figsize)
    all_means = []

    for i, key in enumerate(order):
        if key not in agg:
            continue
        cfg    = key_config.get(key, {})
        label  = cfg.get("label",  key)
        marker = cfg.get("marker", fallback_markers[i % len(fallback_markers)])
        color  = cfg.get("color",  color_palette[i % len(color_palette)])
        ls     = cfg.get("ls", "-")

        bks = sorted(agg[key], key=_bk_sort)
        xs  = [_bk_sort(bk) for bk in bks]
        ys  = [agg[key][bk][f"{metric}_mean"] for bk in bks]
        es  = [agg[key][bk][f"{metric}_ci95"]  for bk in bks]
        all_means.extend(ys)

        ax.errorbar(xs, ys, yerr=es,
                    marker=marker, color=color, linestyle=ls, linewidth=2.5, markersize=11,
                    capsize=err_capsize, elinewidth=err_lw, capthick=err_capthick,
                    label=label, zorder=2)

    ax.set_xlabel(r"Budget (Portion of Scene)", fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}\\%"))
    ax.set_ylim(*_data_ylim(all_means, metric))
    if metric == "psnr":
        ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.5, zorder=0)
        ax.text(ax.get_xlim()[1], PSNR_SATURATION_DB, r" Saturation ($\geq$60 dB)",
                fontsize=13, color="gray", va="bottom", ha="right")
    ax.legend(loc="best", framealpha=0.9, fontsize=18)
    ax.tick_params(labelsize=18)

    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)

    fig.set_constrained_layout(True)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{out_stem}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(out_dir / f"{out_stem}.eps", format="eps", bbox_inches="tight")
    print(f"Wrote {out_dir}/{out_stem}.{{png,eps}}")
    plt.close(fig)

In [9]:
def plot_bar(agg, order, metric, ylabel, out_path):
    """Cross-scene grouped bars, paper style (tab10 KEY_CONFIG colors, no title/grid)."""
    bks    = sorted({b for k in agg for b in agg[k]}, key=_bk_sort)
    groups = [k for k in order if k in agg]
    n_b, n_g = len(bks), len(groups)
    width  = 0.8 / max(n_g, 1)
    x      = np.arange(n_b)

    fig, ax = plt.subplots(figsize=(max(8.0, n_b * 1.3), 4.8))
    all_means = []
    for i, key in enumerate(groups):
        cfg = KEY_CONFIG.get(key, {})
        y   = [agg[key].get(b, {}).get(f"{metric}_mean", 0.0) for b in bks]
        err = [agg[key].get(b, {}).get(f"{metric}_ci95",  0.0) for b in bks]
        all_means.extend(v for b, v in zip(bks, y) if b in agg[key])
        offset = (i - n_g / 2 + 0.5) * width
        ax.bar(x + offset, y, width * 0.92, yerr=err, capsize=err_capsize,
               color=cfg.get("color", color_palette[i % len(color_palette)]),
               label=cfg.get("label", key),
               error_kw={"elinewidth": err_lw, "capthick": err_capthick})

    ax.set_ylim(*_data_ylim(all_means, metric))
    ax.set_xticks(x)
    ax.set_xticklabels([b.replace("%", r"\%") for b in bks])
    ax.set_xlabel(r"Budget (Portion of Scene)", fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.tick_params(labelsize=18)
    ax.legend(fontsize=16, framealpha=0.9)
    ax.set_axisbelow(True)
    if metric == "psnr":
        ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.5, zorder=0)
        ax.text(n_b - 0.5, PSNR_SATURATION_DB, r"Saturation ($\geq$60 dB) ",
                fontsize=13, color="gray", va="bottom", ha="right")

    fig.set_constrained_layout(True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

In [10]:
def plot_grid(scenes_agg, order, metric, ylabel, out_path, ncols=5, key_config=None):
    """Per-scene subplot grid, paper style (tab10 colors, shared top legend, no clutter).
    `scenes_agg` keys are panel titles; `key_config` defaults to KEY_CONFIG (scheme
    lines) -- pass GRID_CONFIG when lines represent grid size instead of scheme."""
    key_config = KEY_CONFIG if key_config is None else key_config
    scenes = list(scenes_agg.keys())
    n      = len(scenes)
    ncols  = min(ncols, n)
    nrows  = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.4, nrows * 3.0),
                             squeeze=False, sharey=True, constrained_layout=True)
    axes_flat = axes.flatten()
    for ax in axes_flat[n:]:
        ax.set_visible(False)

    ylim = _data_ylim(
        [cell[f"{metric}_mean"]
         for a in scenes_agg.values()
         for budgets in a.values()
         for cell in budgets.values()],
        metric,
    )

    for idx, scene in enumerate(scenes):
        ax = axes_flat[idx]
        a  = scenes_agg[scene]
        bks = next((sorted(a[k], key=_bk_sort) for k in order if k in a), None)
        if bks is None:
            continue
        xs = [_bk_sort(b) for b in bks]
        for key in (k for k in order if k in a):
            cfg = key_config.get(key, {})
            y   = [a[key].get(b, {}).get(f"{metric}_mean", float("nan")) for b in bks]
            err = [a[key].get(b, {}).get(f"{metric}_ci95", 0.0) for b in bks]
            ax.errorbar(xs, y, yerr=err,
                        marker=cfg.get("marker", "o"),
                        color=cfg.get("color", color_palette[0]),
                        linestyle=cfg.get("ls", "-"),
                        linewidth=2, markersize=5, capsize=err_capsize,
                        label=cfg.get("label", key))
        ax.set_title(scene.capitalize(), fontsize=14)
        ax.set_ylim(*ylim)
        if metric == "psnr":
            ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.0, zorder=0)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}\\%"))
        ax.tick_params(labelsize=12)
        if idx % ncols == 0:
            ax.set_ylabel(ylabel, fontsize=14)

    # one shared legend on top, collected across panels
    seen = {}
    for ax in axes_flat[:n]:
        for h, l in zip(*ax.get_legend_handles_labels()):
            seen.setdefault(l, h)
    fig.legend(list(seen.values()), list(seen.keys()),
               loc="upper center", ncol=len(seen), fontsize=14,
               framealpha=0.9, bbox_to_anchor=(0.5, 1.12))
    fig.supxlabel(r"Budget (Portion of Scene)", fontsize=16)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

In [11]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

# 1. aggregate line plot (paper main figure)
plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)
plot_metric(agg, order, "lpips", r"LPIPS (Lower Is Better)", "lpips_vs_budget", out_dir)

# 2. per-scene grid
plot_grid(scenes_agg, order, "psnr", r"Quality in PSNR (dB)", grid_dir / "psnr_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "ssim", r"Quality in SSIM", grid_dir / "ssim_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "lpips", r"LPIPS (Lower Is Better)", grid_dir / "lpips_vs_budget.png", ncols=5)
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/exp1_weights/_grid/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/_grid/ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/_grid/lpips_vs_budget.png
Done.


## Exp 2 — Scheme Ranking

Slice of the comprehensive Exp4 grid sweep (grid8, tile_partial packing, screen_area
weight_mode, scheme as the swept dimension) -- not a separate run. Source:
`output/0715/quality_sweep/summary_all.csv` -- post gsorder-bugfix full rerun
(2026-07-15/16; supersedes `../VCUTS_backup/output/0704/.../summary_all_ml_fixed.csv`
(archived 2026-07-17), which had `vd_lod`
wrongly weight-ordered inside tile_partial/tile_strict packing, see PLAN.md "Gsorder bug").
Schemes: `vd_lod` (Tile-Based), `v_lod_w` (heuristic, ours), `ml` (learned, ours),
`oracle_loo` (upper bound). 10 scenes x 150 eval cams.


In [12]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# 2026-07-13: exp2 is a slice of the comprehensive Exp4 grid sweep (grid8, tile_partial
# packing, screen_area weight_mode, scheme as the swept dimension) -- not a separate run.
# 2026-07-16: repointed to the post-gsorder-bugfix rerun (output/0715/) -- the old
# old (now-archived) 0704 CSV had `vd_lod` wrongly weight-ordered inside tile_partial packing, exactly the
# scheme/packing combo this experiment plots. See PLAN.md "Gsorder bug".
SUMMARY_CSV = "output/0715/quality_sweep/summary_all.csv"
OUT_DIR     = "plotting/paper/exp2/"
GROUP_BY    = "scheme"
EXCLUDE_KEYS = ["vd_lod_w"]  # retired 2026-07-03 -- v_lod_w is canonical "ours" heuristic
SLICE_FILTERS = {"grid_shape": "[8, 8, 8]", "packing_mode": "tile_partial", "weight_mode": "screen_area"}

BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)
for _col, _val in SLICE_FILTERS.items():
    df = df[df[_col] == _val]
df = df[~df[GROUP_BY].isin(EXCLUDE_KEYS)].copy()

# provenance: exact sliced rows behind this experiment's figures, alongside the PNGs
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")


Loaded 48000 rows | groups: ['ml', 'oracle_loo', 'v_lod_w', 'vd_lod'] | budgets: ['10%', '25%', '40%', '55%', '70%', '85%', '99%', '100%']


In [13]:
agg    = _aggregate(rows, GROUP_BY)
pm_order, pm_labels = _resolve_order_and_labels(agg, GROUP_BY)

preferred = KEY_ORDER.get(GROUP_BY, pm_order)
order = [k for k in preferred if k in agg] + [k for k in pm_order if k not in preferred and k in agg]

labels = {k: KEY_CONFIG[k]["label"] if k in KEY_CONFIG else pm_labels.get(k, k) for k in order}

_by_scene = _dd(list)
for r in rows:
    _by_scene[r["scene"]].append(r)
scenes_agg = {s: _aggregate(_by_scene[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene}

print(f"order: {order}")
print(f"scenes: {list(scenes_agg.keys())}")

order: ['vd_lod', 'v_lod_w', 'ml', 'oracle_loo']
scenes: ['bicycle', 'garden', 'stump', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship']


In [14]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

# 1. aggregate line plot (paper main figure)
plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)
plot_metric(agg, order, "lpips", r"LPIPS (Lower Is Better)", "lpips_vs_budget", out_dir)

# 2. per-scene grid
plot_grid(scenes_agg, order, "psnr", r"Quality in PSNR (dB)", grid_dir / "psnr_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "ssim", r"Quality in SSIM", grid_dir / "ssim_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "lpips", r"LPIPS (Lower Is Better)", grid_dir / "lpips_vs_budget.png", ncols=5)
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp2/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp2/ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/exp2/_grid/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp2/_grid/ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp2/_grid/lpips_vs_budget.png
Done.


## Exp 3 — GS ordering vs Tiled selection (established 4-way comparison)

Source: `output/0715/quality_sweep/summary_all.csv`, same comprehensive sweep as
Exp2/Exp4 (post gsorder-bugfix rerun, 2026-07-15/16; Exp1 stays on the old
`../VCUTS_backup/output/0704/.../summary_all_ml_fixed.csv` (archived 2026-07-17) since its progressive-packing/vd_lod slice
was never affected by the bug and 0715 lacks the volume/volume_over_d2 weight_mode rows). Established pattern (originally `output/quickplots/0606/exp123_merged/`,
deleted 2026-06-28 per PLAN.md's wrap-up log; `condition` KEY_CONFIG/KEY_ORDER entries and
`experiments/plot_metrics.py`'s `_SCHEME_LABELS` `prog_no_cull`/`prog_culled`/`oracle_tp`/`ml_tp`
kept the naming alive) -- 4 fixed conditions, each a specific (grid, packing_mode, scheme)
point, not a raw column slice:

- `prog_cull`   = grid8,  progressive,   vd_lod      (GS ordering, culled)
- `prog_no_cull`= grid1,  progressive,   vd_lod      (GS ordering, not culled)
- `tiled_oracle`= grid8,  tile_partial,  oracle_loo  (tiled, oracle upper bound)
- `tiled_ml`    = grid8,  tile_partial,  ml          (tiled, learned)


In [15]:
# ── I/O ──────────────────────────────────────────────────────────────────────
SUMMARY_CSV = "output/0715/quality_sweep/summary_all.csv"  # post gsorder-bugfix rerun, 2026-07-16
OUT_DIR     = "plotting/paper/exp3_gs_vs_tiled/"
GROUP_BY    = "condition"
EXCLUDE_KEYS = []

BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

# 4 fixed (grid, packing_mode, scheme) points, flattened into one "condition" column --
# matches the established prog_cull/prog_no_cull/tiled_oracle/tiled_ml naming (KEY_CONFIG
# above, experiments/plot_metrics.py's _SCHEME_LABELS).
EXP3_CONDITIONS = {
    "prog_cull":    ("[8, 8, 8]", "progressive",  "vd_lod"),
    "prog_no_cull": ("[1, 1, 1]", "progressive",  "vd_lod"),
    "tiled_oracle": ("[8, 8, 8]", "tile_partial", "oracle_loo"),
    "tiled_ml":     ("[8, 8, 8]", "tile_partial", "ml"),
}

summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df_full = pd.read_csv(summary_csv)
df_full = df_full[df_full["weight_mode"] == "screen_area"]

parts = []
for cond, (grid, packing, scheme) in EXP3_CONDITIONS.items():
    sub = df_full[(df_full["grid_shape"] == grid) & (df_full["packing_mode"] == packing)
                  & (df_full["scheme"] == scheme)].copy()
    sub["condition"] = cond
    parts.append(sub)
df = pd.concat(parts, ignore_index=True)

# provenance: exact sliced rows behind this experiment's figures, alongside the PNGs
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")
for cond, (grid, packing, scheme) in EXP3_CONDITIONS.items():
    n = sum(1 for r in rows if r["condition"] == cond)
    print(f"  {cond:14s} = grid{grid} {packing:14s} {scheme:10s} -> {n} rows")


Loaded 48000 rows | groups: ['prog_cull', 'prog_no_cull', 'tiled_ml', 'tiled_oracle'] | budgets: ['10%', '25%', '40%', '55%', '70%', '85%', '99%', '100%']
  prog_cull      = grid[8, 8, 8] progressive    vd_lod     -> 12000 rows
  prog_no_cull   = grid[1, 1, 1] progressive    vd_lod     -> 12000 rows
  tiled_oracle   = grid[8, 8, 8] tile_partial   oracle_loo -> 12000 rows
  tiled_ml       = grid[8, 8, 8] tile_partial   ml         -> 12000 rows


In [16]:
agg    = _aggregate(rows, GROUP_BY)
pm_order, pm_labels = _resolve_order_and_labels(agg, GROUP_BY)

preferred = KEY_ORDER.get(GROUP_BY, pm_order)
order = [k for k in preferred if k in agg] + [k for k in pm_order if k not in preferred and k in agg]

labels = {k: KEY_CONFIG[k]["label"] if k in KEY_CONFIG else pm_labels.get(k, k) for k in order}

_by_scene = _dd(list)
for r in rows:
    _by_scene[r["scene"]].append(r)
scenes_agg = {s: _aggregate(_by_scene[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene}

print(f"order: {order}")
print(f"scenes: {list(scenes_agg.keys())}")


order: ['prog_cull', 'prog_no_cull', 'tiled_ml', 'tiled_oracle']
scenes: ['bicycle', 'garden', 'stump', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship']


In [17]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

# 1. aggregate line plot (paper main figure)
plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)
plot_metric(agg, order, "lpips", r"LPIPS (Lower Is Better)", "lpips_vs_budget", out_dir)

# 2. per-scene grid
plot_grid(scenes_agg, order, "psnr", r"Quality in PSNR (dB)", grid_dir / "psnr_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "ssim", r"Quality in SSIM", grid_dir / "ssim_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "lpips", r"LPIPS (Lower Is Better)", grid_dir / "lpips_vs_budget.png", ncols=5)
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp3_gs_vs_tiled/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp3_gs_vs_tiled/ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/exp3_gs_vs_tiled/_grid/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp3_gs_vs_tiled/_grid/ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp3_gs_vs_tiled/_grid/lpips_vs_budget.png
Done.


## Wall-Time / Selection Latency (Exp5)

Per-frame selection cost by condition. Two-stage aggregation (scene -> group), same
principle as the PSNR/SSIM `_aggregate` above: scenes are the independent statistical
unit, not cameras, so CI is computed over `n_scenes`, not pooled cameras.

Source: `time_selection.py` full sweep, `../VCUTS_backup/output/0702/selection_timing/{scene}/{method}/summary.csv`
(150 cams/method; archived 2026-07-13, see PLAN.md's backup-move note). Scene roster is a
static dict (`TIMING_SCENE_GROUP`, not discovered from the sweep directory) — extending to
more scenes means adding a `summary.csv` under an already-listed scene name; scenes without
one yet are skipped with a printed warning.

Timing sweep (`../VCUTS_backup/output/0713/selection_timing_gsorder_weight/`, archived
2026-07-17) predates the 2026-07-14
per-scene-model switch, but that's not a staleness concern here: pooled vs per-scene only
changes how the LGBM model was *fit*, not its inference-time shape/cost -- ms/frame is
unaffected, no rerun needed. Quality-side of the tradeoff plots below (cell computing
`quality_tradeoff_agg`) was repointed to the current 0715 data 2026-07-17 for a different
reason (it needed the post-bicycle-fix rows, not a timing fix).


In [18]:
import matplotlib.patches as mpatches

# ── config ───────────────────────────────────────────────────────────────────
# suffix -> ([sweep dirs, first match wins per scene/method], output dir). "" (grid8,
# canonical) writes to the paper dir with unsuffixed filenames. grid2/grid4 (2026-07-03
# retry, PLAN.md item vii addendum) are an exploratory retry, not paper-canonical -- they
# go to output/quickplots/, not the paper dir.
#
# "" is a 3-dir fallback chain: output/0717/selection_timing_post_optimization, then archived output/0713/selection_timing_gsorder_weight
# (real3 only, --gs-order weight) reran ml/oracle_online with the correct as-deployed
# gaussian_weights cost (test_utility_inmem.py defaults --gs-order weight unconditionally
# for every scheme -- the old 0702 sweep used time_selection.py's --gs-order ply default,
# which skips gaussian_weights for ml/oracle_online and understated their real per-frame
# cost). Falls back to the old 0702 dir for synth scenes (not rerun -- vd_lod/heuristic
# there are unaffected by the flag, and no ml/oracle synth-scene correction was needed for
# this figure). vd_lod/heuristic bicycle/garden/stump ARE picked from the new dir too even
# though unaffected by the flag -- they're faster there for an unrelated reason (greedy_order's
# 2026-07-02 CPU-sort perf fix, selection_core.py:d482e0d, landed same day as the old sweep
# and the old numbers likely predate it -- not a --gs-order effect, just codebase drift).
TIMING_SWEEPS = {
    # 2026-07-17: output/0717/selection_timing_post_optimization prepended -- post
    # matmul-free project_covariance_2d rewrite (utility_calculation.py, 4.68x on the
    # op) + real-budget (not identity-budget) greedy transfer fix (time_selection.py).
    # Covers vd_lod/heuristic/ml_lgbm/progressive_screen_area (prog_screen_area,
    # tiled_vd_lod, tiled_heuristic, tiled_ml) for all 10 scenes -- the 4 conditions
    # that exercise gaussian_weights/greedy. prog_vol_over_d2/tiled_oracle still fall
    # back to 0713/0702 (untouched by this session's changes, not re-run).
    "":       ([Path("output/0717/selection_timing_post_optimization"),
                Path("../VCUTS_backup/output/0713/selection_timing_gsorder_weight"),
                Path("../VCUTS_backup/output/0702/selection_timing")],  Path("plotting/paper/timings_selection")),
    "_grid2": ([Path("../VCUTS_backup/output/0703/selection_timing/grid2")],  Path("output/quickplots/0703_timing_grid_retry")),
    "_grid4": ([Path("../VCUTS_backup/output/0703/selection_timing/grid4")],  Path("output/quickplots/0703_timing_grid_retry")),
}

# full canonical roster (same 10 scenes as SCENE_ORDER above). Only chair/bicycle
# have a timing sweep run so far -- the rest are skipped at load time until they do.
TIMING_SCENE_GROUP = {
    "chair": "synth", "drums": "synth", "ficus": "synth", "hotdog": "synth",
    "materials": "synth", "mic": "synth", "ship": "synth",
    "bicycle": "real", "garden": "real", "stump": "real",
}
TIMING_GROUP_LABELS = {"synth": "Synthetic", "real": "Real"}

# condition -> method dir name (same for every scene, all use canonical "ml_lgbm" now).
# grid2/grid4 lack a tiled_ml sweep (output/ml_models/8/{scene}/AC was trained on grid8
# tiles, not valid at other grid sizes) -- tiled_ml simply won't appear in those bars.
# 2026-07-17 (user): tiled-only, matches exp2's KEY_CONFIG labels exactly -- this figure
# compares deployed selection methods' latency, not the progressive-vs-tiled packing axis.
TIMING_COND_METHOD = {
    "tiled_vd_lod":     "vd_lod",
    "tiled_heuristic":  "heuristic",
    "tiled_ml":         "ml_lgbm",
    "tiled_oracle":     "oracle_online",
}
# ml_rf excluded: ml_lgbm is faster on both scenes measured so far and is the
# canonical ML scheme since 2026-07-02 (PLAN.md).

# same label strings as exp2's KEY_CONFIG (vd_lod/v_lod_w/ml/oracle_loo) -- one naming
# convention across the paper, not a separate "Tiled X" vocabulary for this figure.
TIMING_COND_LABELS = {
    "tiled_vd_lod":     "Tile-Based",
    "tiled_heuristic":  "Heuristic (Ours)",
    "tiled_ml":         "ML (Ours)",
    "tiled_oracle":     "Proxy Oracle",
}
TIMING_COND_ORDER = list(TIMING_COND_METHOD.keys())

# "utility" = tile_weights_s (aggregate per-GS weight -> per-tile W_k) + utility_s
# (evaluate v/d*W_k) -- per time_selection.py these run back-to-back as one
# "produce a per-tile score" step (heuristic: gaussian_weights -> tile_weights ->
# utility). GS Weights stays separate: it's the upstream per-GS signal computation,
# same role as ML Features plays for the ML scheme.
# "render" (oracle's actual LOO renders) is a different operation that happens to
# share the same pipeline slot as "utility" -- kept as its own stage, not merged in.
# colors: reuse color_palette (defined above) by index, not an invented palette.
TIMING_STAGE_DEFS = [
    ("visibility", "Visibility",      color_palette[0]),
    ("gw",         "GS Weights",      color_palette[1]),
    ("ml_feat",    "ML Features",     color_palette[2]),
    ("utility",    "Tile Utility", color_palette[3]),
    ("render",     "LOO Render",      color_palette[4]),
    ("greedy_sel", "Greedy Select",   color_palette[5]),
]

In [19]:
# ── load per-camera timing summaries, one row per (scene, condition), per sweep ────
# each sweep dir is tried in order (first match wins per scene/method); method dir name
# is also tried with a "_wtord" suffix (time_selection.py's --gs-order weight run tags
# ml/oracle_online output dirs this way to avoid colliding with a --gs-order ply run in
# the same tree).
timing_dfs = {}
for _suffix, (_sweep_dirs, _out_dir) in TIMING_SWEEPS.items():
    _timing_rows, _timing_missing = [], []
    for scene, group in TIMING_SCENE_GROUP.items():
        for cond, method in TIMING_COND_METHOD.items():
            csv = None
            for _dir in _sweep_dirs:
                for _mname in (method, f"{method}_wtord"):
                    _cand = _dir / scene / _mname / "summary.csv"
                    if _cand.exists():
                        csv = _cand
                        break
                if csv is not None:
                    break
            if csv is None:
                _timing_missing.append(f"{scene}/{cond}")
                continue
            mdf = pd.read_csv(csv).fillna(0.0)
            _timing_rows.append({
                "scene": scene, "group": group, "condition": cond, "n_cameras": len(mdf),
                "total_ms":   mdf["total_s"].mean() * 1000,
                "visibility": mdf["visibility_s"].mean() * 1000,
                "gw":         mdf["gaussian_weights_s"].mean() * 1000,
                "ml_feat":    (mdf["ml_group_a_s"] + mdf["ml_predict_s"]).mean() * 1000,
                "utility":    (mdf["tile_weights_s"] + mdf["utility_s"]).mean() * 1000,
                "render":     (mdf["full_render_s"] + mdf["tile_renders_s"]).mean() * 1000,
                "greedy_sel": mdf["greedy_s"].mean() * 1000,
            })
    timing_dfs[_suffix] = pd.DataFrame(_timing_rows)
    print(f"[sweep{_suffix or ':grid8'}] loaded {len(_timing_rows)} rows | scenes present: {sorted(timing_dfs[_suffix]['scene'].unique())}")
    if _timing_missing:
        print(f"[sweep{_suffix or ':grid8'}] not swept, skipped: {_timing_missing}")

timing_df = timing_dfs[""]  # kept for backward compat with any ad-hoc inspection below
Path("plotting/paper/timings_selection").mkdir(parents=True, exist_ok=True)
timing_df.to_csv(Path("plotting/paper/timings_selection") / "source_summary.csv", index=False)
timing_df.head()

[sweep:grid8] loaded 40 rows | scenes present: ['bicycle', 'chair', 'drums', 'ficus', 'garden', 'hotdog', 'materials', 'mic', 'ship', 'stump']
[sweep_grid2] loaded 30 rows | scenes present: ['bicycle', 'chair', 'drums', 'ficus', 'garden', 'hotdog', 'materials', 'mic', 'ship', 'stump']
[sweep_grid2] not swept, skipped: ['chair/tiled_ml', 'drums/tiled_ml', 'ficus/tiled_ml', 'hotdog/tiled_ml', 'materials/tiled_ml', 'mic/tiled_ml', 'ship/tiled_ml', 'bicycle/tiled_ml', 'garden/tiled_ml', 'stump/tiled_ml']


[sweep_grid4] loaded 30 rows | scenes present: ['bicycle', 'chair', 'drums', 'ficus', 'garden', 'hotdog', 'materials', 'mic', 'ship', 'stump']
[sweep_grid4] not swept, skipped: ['chair/tiled_ml', 'drums/tiled_ml', 'ficus/tiled_ml', 'hotdog/tiled_ml', 'materials/tiled_ml', 'mic/tiled_ml', 'ship/tiled_ml', 'bicycle/tiled_ml', 'garden/tiled_ml', 'stump/tiled_ml']


,scene,group,condition,n_cameras,total_ms,visibility,gw,ml_feat,utility,render,greedy_sel
0,chair,synth,tiled_vd_lod,150,1.549900,0.531588,0.000000,0.000000,0.151975,0.000000,0.866337
1,chair,synth,tiled_heuristic,150,6.071673,0.563151,4.111820,0.000000,0.469947,0.000000,0.926756
2,chair,synth,tiled_ml,150,15.836689,0.808752,4.859424,9.105539,0.000000,0.000000,1.062975
3,chair,synth,tiled_oracle,150,2194.833552,0.517582,4.986808,0.000000,0.000000,2185.728501,3.600661
4,drums,synth,tiled_vd_lod,150,1.840784,0.583374,0.000000,0.000000,0.160533,0.000000,1.096877


In [20]:
# ── aggregate: group mean per condition (+ CI once >1 scene/group exists), per sweep ──
def _ci95(vals):
    n = len(vals)
    return 1.96 * float(np.std(vals, ddof=1)) / np.sqrt(n) if n > 1 else 0.0


timing_bar_aggs = {}
for _suffix, _df in timing_dfs.items():
    _agg = {"synth": {}, "real": {}}
    for group in _agg:
        gdf = _df[_df["group"] == group]
        for cond in TIMING_COND_ORDER:
            vals = gdf.loc[gdf["condition"] == cond, "total_ms"].to_numpy()
            if len(vals) == 0:
                continue
            _agg[group][cond] = {"mean": float(np.mean(vals)), "ci95": _ci95(vals)}
    timing_bar_aggs[_suffix] = _agg

timing_bar_agg = timing_bar_aggs[""]  # kept for backward compat with any ad-hoc inspection below


def timing_stage_group_means(group, df=None):
    """Per-condition stage-mean ms, averaged across scenes in `group`. `df` defaults to
    the grid8 sweep for backward compat; pass timing_dfs[suffix] for other sweeps."""
    gdf = (df if df is not None else timing_df)
    gdf = gdf[gdf["group"] == group]
    return {
        cond: {sk: float(gdf.loc[gdf["condition"] == cond, sk].mean()) for sk, _, _ in TIMING_STAGE_DEFS}
        for cond in TIMING_COND_ORDER if cond in gdf["condition"].unique()
    }


print(timing_bar_agg)

{'synth': {'tiled_vd_lod': {'mean': 1.7161656082385564, 'ci95': np.float64(0.14082197148252043)}, 'tiled_heuristic': {'mean': 7.469943320949824, 'ci95': np.float64(0.7324352584251907)}, 'tiled_ml': {'mean': 16.77703175133178, 'ci95': np.float64(1.833004027538037)}, 'tiled_oracle': {'mean': 2608.4480643715883, 'ci95': np.float64(1226.4894064500538)}}, 'real': {'tiled_vd_lod': {'mean': 60.67999319950735, 'ci95': np.float64(10.759572739771546)}, 'tiled_heuristic': {'mean': 225.0233757305807, 'ci95': np.float64(35.759862450114184)}, 'tiled_ml': {'mean': 212.18763134708004, 'ci95': np.float64(35.31234002784133)}, 'tiled_oracle': {'mean': 604.4919144031073, 'ci95': np.float64(133.75906059199818)}}}


In [21]:
def plot_timing_bar(bar_agg_group, cond_order, cond_labels, color, label, out_path, ylim):
    """Single-group bar, log y-axis (latency spans ~3 orders of magnitude: oracle vs the
    rest, in both synth and real). `ylim` is shared across synth/real calls so the two
    figures are visually comparable on the same scale."""
    conds = [c for c in cond_order if c in bar_agg_group]
    n = len(conds)
    x = np.arange(n)
    means = [bar_agg_group[c].get("mean", np.nan) for c in conds]
    cis   = [bar_agg_group[c].get("ci95", 0.0) for c in conds]

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.bar(x, means, 0.6, yerr=cis,
           error_kw={"elinewidth": err_lw, "capthick": err_capthick, "capsize": err_capsize},
           color=color, label=label)

    ax.set_yscale("log")
    ax.set_ylim(*ylim)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v)}" if v >= 1 else f"{v:.1f}"))
    ax.set_xticks(x)
    ax.set_xticklabels([cond_labels[c] for c in conds], fontsize=17)
    ax.set_ylabel(r"Selection Latency (ms/frame)", fontsize=18)
    ax.tick_params(labelsize=19)

    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)
    ax.legend(fontsize=18, framealpha=0.9, loc="upper left")

    fig.set_constrained_layout(True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)


def plot_timing_stages(stage_means_by_cond, cond_order, cond_labels, out_path):
    """Per-stage stacked bar with a broken y-axis (oracle ~2500ms dwarfs the rest, <300ms).
    Left/right spines stay closed on both panels (standard broken-axis convention) --
    only the shared top/bottom seam is cut, marked by the diagonal break ticks. No
    figure title (synth vs. real is in the filename/caption, not baked into the PNG)."""
    conds = [c for c in cond_order if c in stage_means_by_cond]
    totals = [sum(stage_means_by_cond[c].values()) for c in conds]
    non_oracle_max = max(t for c, t in zip(conds, totals) if c != "tiled_oracle")
    break_y = non_oracle_max * 1.35
    top_max = max(totals) * 1.22  # extra headroom so the top value label doesn't clip the spine

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(7.5, 6.0),
        gridspec_kw={"height_ratios": [1, 2.2], "hspace": 0.08},
        constrained_layout=False,
    )

    n = len(conds)
    x = np.arange(n)
    w = 0.55
    for ax in (ax_top, ax_bot):
        bottoms = np.zeros(n)
        for sk, label, color in TIMING_STAGE_DEFS:
            heights = np.array([stage_means_by_cond[c][sk] for c in conds])
            ax.bar(x, heights, w, bottom=bottoms, color=color, label=label)
            bottoms += heights

    ax_bot.set_ylim(0, break_y)
    ax_top.set_ylim(break_y, top_max)

    for i, total in enumerate(totals):
        if total > break_y:
            ax, off = ax_top, (top_max - break_y) * 0.03
        else:
            ax, off = ax_bot, break_y * 0.02
        ax.text(i, total + off, f"{total:.1f}", ha="center", va="bottom", fontsize=14, color="#333333")

    ax_bot.set_xticks(x)
    ax_bot.set_xticklabels([cond_labels[c] for c in conds], fontsize=16)
    ax_bot.set_ylabel(r"Selection Latency (ms/frame)", fontsize=17)
    ax_bot.tick_params(axis="y", labelsize=16)
    ax_top.tick_params(axis="y", labelsize=16)
    ax_bot.tick_params(axis="x", length=0)

    ax_top.spines["bottom"].set_visible(False)
    ax_bot.spines["top"].set_visible(False)
    ax_top.tick_params(axis="x", bottom=False, labelbottom=False)

    d = 0.012
    kw = dict(transform=ax_top.transAxes, color="k", clip_on=False, linewidth=1.2)
    ax_top.plot((-d, +d), (-2 * d, +2 * d), **kw)
    kw = dict(transform=ax_bot.transAxes, color="k", clip_on=False, linewidth=1.2)
    ax_bot.plot((-d, +d), (1 - d, 1 + d), **kw)

    # legend goes in ax_top's blank region (only the rightmost/oracle bar has content there)
    handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in TIMING_STAGE_DEFS]
    ax_top.legend(handles=handles, loc="upper left", ncol=2, fontsize=13, framealpha=0.9)

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

In [22]:
# ylim computed per-sweep (not shared across grid8/grid2/grid4 -- grid2/4's dynamic
# range is much smaller than grid8's, sharing grid8's scale would flatten the bars).
for _suffix, _bar_agg in timing_bar_aggs.items():
    _out_dir = TIMING_SWEEPS[_suffix][1]
    _all_lo = [v["mean"] - v["ci95"] for g in _bar_agg.values() for v in g.values()]
    _all_hi = [v["mean"] + v["ci95"] for g in _bar_agg.values() for v in g.values()]
    _ylim = (min(_all_lo) * 0.7, max(_all_hi) * 1.4)

    for group, title in TIMING_GROUP_LABELS.items():
        plot_timing_bar(_bar_agg[group], TIMING_COND_ORDER, TIMING_COND_LABELS,
                         color_palette[0], title, _out_dir / f"selection_timing_bar_{group}{_suffix}.png",
                         ylim=_ylim)

        stage_means = timing_stage_group_means(group, df=timing_dfs[_suffix])
        plot_timing_stages(stage_means, TIMING_COND_ORDER, TIMING_COND_LABELS,
                            _out_dir / f"selection_timing_stages_{group}{_suffix}.png")

print("Done.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/timings_selection/selection_timing_bar_synth.png


/tmp/ipykernel_3779571/1798321835.py:95: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/timings_selection/selection_timing_stages_synth.png


...

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0703_timing_grid_retry/selection_timing_bar_real_grid4.png


/tmp/ipykernel_3779571/1798321835.py:95: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0703_timing_grid_retry/selection_timing_stages_real_grid4.png
Done.


### Selection time vs. scene size, and quality/latency tradeoff

Two additional views of the same Exp5 timing data (`timing_dfs[""]`, grid8):
1. Latency vs. `#Gaussians` per (scene, condition) -- does selection cost scale with
   scene size, and how does that scaling differ by condition?
2. PSNR (at the established `@40%` headline budget, see PLAN.md) vs. latency, per
   condition, real/synth split -- the quality/latency tradeoff each condition buys.

`SCENE_N_GS` is read once from each scene's `point_cloud.ply` header (`element vertex
<N>`) and hardcoded here, same static-dict pattern as `TIMING_SCENE_GROUP` above --
avoids a hard dependency on `exp-dataset/` being mounted just to make this plot.

In [23]:
SCENE_N_GS = {
    "chair": 258406, "drums": 351952, "ficus": 293580, "hotdog": 148783,
    "materials": 285916, "mic": 307307, "ship": 325316,
    "bicycle": 5998962, "garden": 5834784, "stump": 4961797,
}

def plot_time_vs_ngs(df, out_path):
    fig, ax = plt.subplots(figsize=(7.0, 5.2))
    markers = ["o", "s", "^", "D", "P", "*"]
    for i, cond in enumerate(TIMING_COND_ORDER):
        sub = df[df["condition"] == cond]
        if sub.empty:
            continue
        xs = [SCENE_N_GS[s] for s in sub["scene"]]
        ys = sub["total_ms"].to_numpy()
        ax.scatter(xs, ys, marker=markers[i % len(markers)],
                   color=color_palette[i % len(color_palette)],
                   s=130, label=TIMING_COND_LABELS[cond].replace("\n", " "), zorder=3)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"Scene Size (\# Gaussians)", fontsize=18)
    ax.set_ylabel(r"Selection Latency (ms/frame)", fontsize=18)
    ax.tick_params(labelsize=15)
    ax.legend(fontsize=14, framealpha=0.9, ncol=2, loc="lower right")
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)

    fig.set_constrained_layout(True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)


plot_time_vs_ngs(timing_dfs[""], Path("plotting/paper/timings_selection/selection_time_vs_ngs.png"))


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/timings_selection/selection_time_vs_ngs.png


In [24]:
# condition -> (grid_shape, packing_mode, scheme, weight_mode) in the quality CSV --
# same 6 conditions as TIMING_COND_METHOD, mapped onto the (grid,packing,scheme,weight)
# points already established elsewhere in this notebook (EXP3_CONDITIONS / exp1's
# weight-mode slice / exp2's scheme slice), not a new/independent mapping.
TIMING_COND_QUALITY = {
    "prog_screen_area": ("[8, 8, 8]", "progressive",  "vd_lod",     "screen_area"),
    "prog_vol_over_d2": ("[8, 8, 8]", "progressive",  "vd_lod",     "volume_over_d2"),
    "tiled_vd_lod":     ("[8, 8, 8]", "tile_partial", "vd_lod",     "screen_area"),
    "tiled_heuristic":  ("[8, 8, 8]", "tile_partial", "v_lod_w",    "screen_area"),
    "tiled_ml":         ("[8, 8, 8]", "tile_partial", "ml",         "screen_area"),
    "tiled_oracle":     ("[8, 8, 8]", "tile_partial", "oracle_loo", "screen_area"),
}
TRADEOFF_BUDGET_PCT = "40%"  # PLAN.md's established headline comparison point
_TRADEOFF_BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]
TRADEOFF_METRICS = ["psnr", "ssim", "lpips"]

# 2026-07-17: all 4 tile_partial/grid8/screen_area conditions (tiled_vd_lod/heuristic/
# ml/oracle) plus prog_screen_area now come from the post-gsorder-fix, re-aggregated
# 0715 sweep (bicycle's grid8/tile_partial was missing until a re-concat on 2026-07-17 --
# see PLAN.md), which also carries the current per-scene ML model, not the retired pooled
# one. Only prog_vol_over_d2 stays on 0704 -- 0715 doesn't sweep the volume_over_d2
# weight_mode, only screen_area.
_quality_full  = pd.read_csv("../VCUTS_backup/output/0704/quality_sweep/summary_all_ml_fixed.csv")  # archived 2026-07-17
_quality_fixed = pd.read_csv("output/0715/quality_sweep/summary_all.csv")
_tradeoff_parts = []
for cond, (grid, packing, scheme, wmode) in TIMING_COND_QUALITY.items():
    src = _quality_full if cond == "prog_vol_over_d2" else _quality_fixed
    sub = src[(src["grid_shape"] == grid) & (src["packing_mode"] == packing)
              & (src["scheme"] == scheme) & (src["weight_mode"] == wmode)].copy()
    sub["condition"] = cond
    _tradeoff_parts.append(sub)
_tradeoff_df = pd.concat(_tradeoff_parts, ignore_index=True)
Path("plotting/paper/timings_selection").mkdir(parents=True, exist_ok=True)
_tradeoff_df.to_csv(Path("plotting/paper/timings_selection") / "source_tradeoff.csv", index=False)

# pooled across all 10 scenes -- no real/synth split for this plot (unlike the bar chart
# above, which splits because real/synth latency differ by orders of magnitude; here both
# axes are pooled together so the split isn't needed).
_tradeoff_rows = _apply_budget_labels(_tradeoff_df.to_dict("records"), _TRADEOFF_BUDGET_PCTS)
_cond_agg = _aggregate(_tradeoff_rows, "condition")
quality_tradeoff_agg = {
    metric: {
        cond: {"mean": _cond_agg[cond][TRADEOFF_BUDGET_PCT][f"{metric}_mean"],
               "ci95": _cond_agg[cond][TRADEOFF_BUDGET_PCT][f"{metric}_ci95"]}
        for cond in _cond_agg if TRADEOFF_BUDGET_PCT in _cond_agg[cond]
    }
    for metric in TRADEOFF_METRICS
}

# pooled latency, same 10-scene pool (CI over n=10, not per-group n=3/7)
timing_pooled_agg = {}
for cond in TIMING_COND_ORDER:
    vals = timing_dfs[""].loc[timing_dfs[""]["condition"] == cond, "total_ms"].to_numpy()
    if len(vals) == 0:
        continue
    timing_pooled_agg[cond] = {"mean": float(np.mean(vals)), "ci95": _ci95(vals)}

print(list(quality_tradeoff_agg["psnr"].keys()), list(timing_pooled_agg.keys()))


['prog_screen_area', 'prog_vol_over_d2', 'tiled_vd_lod', 'tiled_heuristic', 'tiled_ml', 'tiled_oracle'] ['tiled_vd_lod', 'tiled_heuristic', 'tiled_ml', 'tiled_oracle']


In [ ]:
TRADEOFF_YLABELS = {
    "psnr":  ("PSNR (dB)", False),
    "ssim":  ("SSIM", False),
    "lpips": ("LPIPS", True),  # lower is better
}

# 2026-07-22: run in a subprocess on an EXPLICIT absolute path to the base env's
# python (matplotlib>=3.10), NOT sys.executable/gsquic (matplotlib 3.9.2) -- an
# unqualified "python3" resolves back to gsquic's python here, since conda run -n
# gsquic puts that env's bin/ first on PATH. Confirmed root cause (A/B tested):
# 3.9.2's EPS bbox_inches="tight" + LaTeX + Ghostscript-crop pipeline silently
# produces a structurally-valid but blank EPS for this plot's shape (errorbar with
# BOTH xerr and yerr on a log-x axis, plus the "Better" annotate arrow) --
# reproducible under 3.9.2 regardless of process isolation or data; 3.10.6 renders
# it correctly every time. See plotting/_plot_tradeoff_subprocess.py's docstring.
import json as _json
import subprocess as _subprocess

_tradeoff_input = {
    "timing_pooled_agg": {k: {"mean": float(v["mean"]), "ci95": float(v["ci95"])}
                           for k, v in timing_pooled_agg.items()},
    "quality_tradeoff_agg": {m: {k: {"mean": float(v["mean"]), "ci95": float(v["ci95"])}
                                  for k, v in quality_tradeoff_agg[m].items()}
                              for m in TRADEOFF_METRICS},
    "tradeoff_ylabels": TRADEOFF_YLABELS,
    "cond_order": TIMING_COND_ORDER,
    "cond_labels": TIMING_COND_LABELS,
    "budget_pct": TRADEOFF_BUDGET_PCT,
}
_tradeoff_json = Path("plotting/paper/timings_selection/_tradeoff_input.json")
_tradeoff_json.parent.mkdir(parents=True, exist_ok=True)
_tradeoff_json.write_text(_json.dumps(_tradeoff_input))
_BASE_PY = "/mnt/data1/samk/anaconda3/bin/python3"
_subprocess.run([_BASE_PY, "plotting/_plot_tradeoff_subprocess.py",
                  str(_tradeoff_json), "plotting/paper/timings_selection"], check=True)
print("Done.")


## Exp 4 — Granularity (tile-count sweep)

Source: `output/0715/quality_sweep/summary_all.csv` -- full 10-scene rerun after the
gsorder bug fix (2026-07-15/16, `vd_lod` was wrongly weight-ordered inside
tile_partial/tile_strict packing -- root cause of this experiment's earlier DUBIOUS
status; see PLAN.md "Gsorder bug"). Supersedes the old
`../VCUTS_backup/output/0704/quality_sweep/summary_all_ml_fixed.csv` (archived
2026-07-17; merged: old non-ml rows from the 2026-07-09 consolidated sweep + fixed-label
`ml` rows from `../VCUTS_backup/output/0709/quality_sweep_ml_retrain/` after the
`ml/features.py` label-censoring fix
+ pooled-model retrain -- see PLAN.md Open item 11). grid in {1,2,4,8,16}³, packing in
{progressive, tile_partial, tile_strict}. Progressive only has `vd_lod` (pure GS
ordering, no tiling); tiled packings run all 4 schemes. Same two-stage scene-then-CI
aggregation as Exp1/Exp2 (`_aggregate`), with `grid` as an extra group-by dimension.
PSNR/SSIM/LPIPS all plotted (LPIPS enabled for this sweep via `--lpips`; not yet
available for Exp1/Exp2, which predate the all-experiments-compute-LPIPS rule).

This grid sweep also answers Exp3 3B, no separate rerun needed:
- **3B (culling benefit):** `psnr_vs_budget_progressive.png` / `..._ssim_...` /
  `..._lpips_...` below -- progressive packing's tile-visibility gate only has teeth once
  tiles are smaller than the scene; grid1³ is a single scene-spanning tile (the gate never
  fires, i.e. the "no-cull" limit), grid16³ is the most-culled point on the same line.

**3A (tile_partial vs tile_strict) is no longer answered here** -- the
`_tile_strict_archive/` figures (including the partial-minus-strict delta) were retired
2026-07-17 (tile_strict dropped from the paper 2026-07-14, and its data is mostly unswept
in the current CSV anyway). If 3A is needed for the paper, it needs a fresh tile_strict
sweep + a new figure, not this notebook as-is.


In [26]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# 2026-07-09: ml/features.py had a label-censoring bug (mse_loo<=0 rows silently
# NaN-dropped instead of floored) that starved the ml scheme's training data on real
# scenes at fine grids. Fixed + all pooled models retrained + ml scheme rerun
# (../VCUTS_backup/output/0709/quality_sweep_ml_retrain/, archived 2026-07-17). That merged CSV was in turn superseded
# 2026-07-16 by output/0715/quality_sweep/summary_all.csv after the gsorder bug fix
# (`vd_lod` was wrongly weight-ordered inside tile_partial/tile_strict packing --
# see PLAN.md "Gsorder bug"); this is the current comprehensive, correct Exp4 dataset.
EXP4_SUMMARY_CSV = "output/0715/quality_sweep/summary_all.csv"
EXP4_OUT_DIR     = Path("plotting/paper/exp4_granularity/")
# tile_strict dropped from the paper (2026-07-14, advisor comment); 2026-07-17: stopped
# generating the _tile_strict_archive/ reference figures entirely (tile_strict data is
# also mostly unswept in the current CSV -- only ficus/grid8 present -- so the archive
# figures were misleadingly single-scene anyway). Exp3A (partial vs strict) is now an
# open figure gap, not answered by this notebook -- see PLAN.md if it needs reinstating.
EXP4_BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]
EXP4_SCHEME_ORDER = ["vd_lod", "v_lod_w", "ml", "oracle_loo"]

GRID_ORDER  = [1, 2, 4, 8, 16]
GRID_CONFIG = {
    1:  {"label": r"$Pri_{1^3}$",  "marker": "o", "color": "#d62728"},
    2:  {"label": r"$Pri_{2^3}$",  "marker": "s", "color": "#ff7f0e"},
    4:  {"label": r"$Pri_{4^3}$",  "marker": "^", "color": "#2ca02c"},
    8:  {"label": r"$Pri_{8^3}$",  "marker": "D", "color": "#1f77b4"},
    16: {"label": r"$Pri_{16^3}$", "marker": "P", "color": "#9467bd"},
}

exp4_csv = Path(EXP4_SUMMARY_CSV)
if not exp4_csv.exists():
    raise FileNotFoundError(f"{exp4_csv}  (cwd={Path.cwd()})")

exp4_df = pd.read_csv(exp4_csv)
# exp1's weight-mode slice reuses grid8/progressive/vd_lod (volume/volume_over_d2/random
# on top of screen_area) -- keep only screen_area or grid8/progressive is 4x-inflated.
exp4_df = exp4_df[exp4_df["weight_mode"] == "screen_area"].copy()
exp4_df["grid"] = exp4_df["grid_shape"].str.extract(r"\[(\d+)").astype(int)

# provenance: exact sliced rows behind this experiment's figures, alongside the PNGs
EXP4_OUT_DIR.mkdir(parents=True, exist_ok=True)
exp4_df.to_csv(EXP4_OUT_DIR / "source_summary.csv", index=False)

exp4_rows = exp4_df.to_dict("records")
exp4_rows = _apply_budget_labels(exp4_rows, EXP4_BUDGET_PCTS)

print(f"Loaded {len(exp4_rows)} rows | packing_mode: {sorted(exp4_df['packing_mode'].unique())} "
      f"| grid: {sorted(exp4_df['grid'].unique())} | scheme: {sorted(exp4_df['scheme'].unique())}")


Loaded 304800 rows | packing_mode: ['progressive', 'tile_partial', 'tile_strict'] | grid: [np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16)] | scheme: ['ml', 'oracle_loo', 'v_lod_w', 'vd_lod']


In [27]:
def _exp4_agg_by_grid(rows, packing_mode, scheme=None):
    """Filter to one packing_mode (+ optional scheme), aggregate grouped by grid."""
    filtered = [r for r in rows if r["packing_mode"] == packing_mode
                and (scheme is None or r["scheme"] == scheme)]
    return _aggregate(filtered, "grid")


def _exp4_agg_by_scheme(rows, packing_mode, grid):
    """Filter to one (packing_mode, grid), aggregate grouped by scheme."""
    filtered = [r for r in rows if r["packing_mode"] == packing_mode and r["grid"] == grid]
    return _aggregate(filtered, "scheme")


EXP4_OUT_DIR.mkdir(parents=True, exist_ok=True)
(EXP4_OUT_DIR / "_grid").mkdir(parents=True, exist_ok=True)

# 1. Progressive (pure GS ordering, no tiling) -- single panel, lines = grid size
prog_agg = _exp4_agg_by_grid(exp4_rows, "progressive")
plot_metric(prog_agg, GRID_ORDER, "psnr", r"Quality in PSNR (dB)",
            "psnr_vs_budget_progressive", EXP4_OUT_DIR, key_config=GRID_CONFIG)
plot_metric(prog_agg, GRID_ORDER, "ssim", r"Quality in SSIM",
            "ssim_vs_budget_progressive", EXP4_OUT_DIR, key_config=GRID_CONFIG)
plot_metric(prog_agg, GRID_ORDER, "lpips", r"LPIPS (Lower Is Better)",
            "lpips_vs_budget_progressive", EXP4_OUT_DIR, key_config=GRID_CONFIG)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/psnr_vs_budget_progressive.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/ssim_vs_budget_progressive.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/lpips_vs_budget_progressive.{png,eps}


In [28]:
# Single-panel Tiled ML (Learned, ours) only, lines=grid size, tile_partial packing --
# same data/style as the "Learned (ours)" panel of _grid/psnr_vs_budget_tile_partial.png,
# pulled out standalone (flat in EXP4_OUT_DIR, matching the flat quickplots/0629_exp4
# naming convention, not nested under _grid/).
ml_by_grid = _exp4_agg_by_grid(exp4_rows, "tile_partial", scheme="ml")
plot_metric(ml_by_grid, GRID_ORDER, "psnr", r"Quality in PSNR (dB)",
            "tiled_ml_psnr_vs_budget", EXP4_OUT_DIR, key_config=GRID_CONFIG)
plot_metric(ml_by_grid, GRID_ORDER, "ssim", r"Quality in SSIM",
            "tiled_ml_ssim_vs_budget", EXP4_OUT_DIR, key_config=GRID_CONFIG)
plot_metric(ml_by_grid, GRID_ORDER, "lpips", r"LPIPS (Lower Is Better)",
            "tiled_ml_lpips_vs_budget", EXP4_OUT_DIR, key_config=GRID_CONFIG)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/tiled_ml_psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/tiled_ml_ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/tiled_ml_lpips_vs_budget.{png,eps}


In [29]:
# 2. Tiled packing (tile_partial only -- tile_strict dropped from the paper 2026-07-14,
# _tile_strict_archive/ figures retired 2026-07-17). One panel per scheme, lines = grid size.
panels_agg = {
    KEY_CONFIG[s]["label"]: _exp4_agg_by_grid(exp4_rows, "tile_partial", scheme=s)
    for s in EXP4_SCHEME_ORDER
}
plot_grid(panels_agg, GRID_ORDER, "psnr", r"Quality in PSNR (dB)",
          EXP4_OUT_DIR / "_grid" / "psnr_vs_budget_tile_partial.png",
          ncols=4, key_config=GRID_CONFIG)
plot_grid(panels_agg, GRID_ORDER, "ssim", r"Quality in SSIM",
          EXP4_OUT_DIR / "_grid" / "ssim_vs_budget_tile_partial.png",
          ncols=4, key_config=GRID_CONFIG)
plot_grid(panels_agg, GRID_ORDER, "lpips", r"LPIPS (Lower Is Better)",
          EXP4_OUT_DIR / "_grid" / "lpips_vs_budget_tile_partial.png",
          ncols=4, key_config=GRID_CONFIG)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/psnr_vs_budget_tile_partial.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/ssim_vs_budget_tile_partial.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/lpips_vs_budget_tile_partial.png


In [30]:
# 3. 8^3 vs 16^3 head-to-head (tile_partial only) -- one panel per grid, lines = scheme.
panels_agg = {
    GRID_CONFIG[g]["label"]: _exp4_agg_by_scheme(exp4_rows, "tile_partial", g)
    for g in (8, 16)
}
plot_grid(panels_agg, EXP4_SCHEME_ORDER, "psnr", r"Quality in PSNR (dB)",
          EXP4_OUT_DIR / "_grid" / "psnr_8v16_tile_partial.png", ncols=2)
plot_grid(panels_agg, EXP4_SCHEME_ORDER, "ssim", r"Quality in SSIM",
          EXP4_OUT_DIR / "_grid" / "ssim_8v16_tile_partial.png", ncols=2)
plot_grid(panels_agg, EXP4_SCHEME_ORDER, "lpips", r"LPIPS (Lower Is Better)",
          EXP4_OUT_DIR / "_grid" / "lpips_8v16_tile_partial.png", ncols=2)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/psnr_8v16_tile_partial.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/ssim_8v16_tile_partial.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/lpips_8v16_tile_partial.png


In [31]:
# 5. Grouped bar: x=grid size, bars=tile_partial schemes, one panel per (metric, budget).
# Same aggregation convention as everywhere else (two-stage scene-then-group CI via
# _aggregate), same visual style as plot_bar above (grouped bars, KEY_CONFIG colors/labels).
EXP4_BAR_DIR = EXP4_OUT_DIR / "_bar_by_grid"


def plot_bar_by_grid(rows, metric, ylabel, budget_pct, out_path):
    budget_key = f"{budget_pct}%"
    schemes = EXP4_SCHEME_ORDER
    grids = GRID_ORDER
    data = {}
    for g in grids:
        filtered = [r for r in rows if r["packing_mode"] == "tile_partial"
                    and r["grid"] == g and r["_budget_key"] == budget_key]
        agg = _aggregate(filtered, "scheme")
        for s in schemes:
            data.setdefault(s, {})[g] = agg.get(s, {}).get(budget_key, {})

    n_g, n_s = len(grids), len(schemes)
    width = 0.8 / n_s
    x = np.arange(n_g)
    fig, ax = plt.subplots(figsize=(max(8.0, n_g * 1.3), 4.8))
    all_means = []
    for i, s in enumerate(schemes):
        cfg = KEY_CONFIG[s]
        y   = [data[s][g].get(f"{metric}_mean", 0.0) for g in grids]
        err = [data[s][g].get(f"{metric}_ci95", 0.0) for g in grids]
        all_means.extend(v for g, v in zip(grids, y) if data[s][g])
        offset = (i - n_s / 2 + 0.5) * width
        ax.bar(x + offset, y, width * 0.92, yerr=err, capsize=err_capsize,
               color=cfg["color"], label=cfg["label"],
               error_kw={"elinewidth": err_lw, "capthick": err_capthick})

    ax.set_ylim(*_data_ylim(all_means, metric))
    ax.set_xticks(x)
    ax.set_xticklabels([GRID_CONFIG[g]["label"] for g in grids])
    ax.set_xlabel(r"Grid Size", fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.tick_params(labelsize=18)
    ax.legend(fontsize=16, framealpha=0.9)
    ax.set_axisbelow(True)
    if metric == "psnr":
        ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.5, zorder=0)
        ax.text(n_g - 0.5, PSNR_SATURATION_DB, r"Saturation ($\geq$60 dB) ",
                fontsize=13, color="gray", va="bottom", ha="right")

    fig.set_constrained_layout(True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)


for budget_pct in [25, 40, 55]:
    for metric, ylabel in [("psnr", r"Quality in PSNR (dB)"), ("ssim", r"Quality in SSIM"),
                            ("lpips", r"LPIPS (Lower Is Better)")]:
        plot_bar_by_grid(exp4_rows, metric, ylabel, budget_pct,
                          EXP4_BAR_DIR / f"{metric}_bar_budget{budget_pct}.png")
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_bar_by_grid/psnr_bar_budget25.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_bar_by_grid/ssim_bar_budget25.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/exp4_granularity/_bar_by_grid/psnr_bar_budget55.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_bar_by_grid/ssim_bar_budget55.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_bar_by_grid/lpips_bar_budget55.png
Done.


## ML scheme comparison — pooled-all10 vs domain-split vs per-scene (2026-07-13, per-scene arm added)

Grid8/tile_partial/screen_area only. 5 canonical arms, same set/color/label everywhere below
(budget-line plots and CDFs): `oracle_loo` (LOO-MSE upper bound, not deployable, ceiling only),
`reg_all` (deployed `pooled_all10_ACG` regressor), `reg_split` (domain-split regressor --
real3_reg on real3 scenes + synth7_reg on synth7 scenes, ONE method not two, see below),
`reg_per_scene` (10 individually-trained per-scene regressors, one method not ten, same
relabel-at-load-time trick as `reg_split`), `vd_lod` (deployed geometry-only baseline, no ML).

**LGBMRanker arms removed.** The rank track (`rank_all`/`rank_split`) was formally rejected
2026-07-13 after a fair post-bugfix retest -- every rank arm lost to its domain-matched
regressor arm on its own domain, see `.claude/concluded_experiments.md`. Its sweep output was
deleted as part of that rejection; this section no longer references it.

**Domain-split / per-scene are each one method, not many.** `real3_reg`/`synth7_reg` relabel
to `reg_split`; all 10 `per_scene/<scene>` runs relabel to `reg_per_scene` -- each source CSV
only has rows for its own scene(s), so both arms naturally resolve to "whichever
domain/scene-matched model applies" in every slice below with zero special-casing: pooled
across all 10 scenes the parts just concatenate; sliced to real3-only or synth7-only, only
the matching rows survive the filter.

Every figure below has 3 panels in one file: **all** (10 scenes) / **real3** / **synth7**.
1. **Budget-line plots** (`{psnr,ssim,lpips}_vs_budget.png`) — mean ± 95% CI per budget,
   matched by percentile label. Per-scene small multiples in `_grid/` (real3/synth7 only --
   an "all" grid would just be their union).
2. **CDF** (`cdf.png`) — empirical CDF of per-(scene,camera) PSNR at a single fixed budget
   (40%, the standard reference point used throughout this project's candidate comparisons).

**Read direction (CDF only):** lower / further-right at a given PSNR = better. Oracle should
sit lowest in every panel.


In [32]:
# ── I/O ──────────────────────────────────────────────────────────────────────
OUT_DIR     = "output/quickplots/rank_vs_pool_comparison/"  # exploratory, not paper-canonical yet
GROUP_BY    = "model_arm"
BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

REAL3_SCENES  = ["bicycle", "garden", "stump"]
SYNTH7_SCENES = ["chair", "drums", "ficus", "hotdog", "materials", "mic", "ship"]

# domain-split relabels to ONE canonical arm name (reg_split / rank_split) -- see markdown
# above for why this makes every downstream slice (all/real3/synth7) work with no special-casing.
# rank arms removed 2026-07-13 (formally rejected, sweep output deleted -- see rvpmd01 above)
ARM_CSVS = {
    "../VCUTS_backup/output/0709/quality_sweep_ml_retrain/summary_all.csv":            "reg_all",  # archived 2026-07-17
    "../VCUTS_backup/output/0713/rank_vs_pool_comparison/real3_reg/summary_all.csv":   "reg_split",
    "../VCUTS_backup/output/0713/rank_vs_pool_comparison/synth7_reg/summary_all.csv":  "reg_split",
    # 10 individually-trained per-scene models (2026-07-13), one summary_all.csv with all
    # 10 scenes already merged by exp_ml_per_scene_sweep.sh's concat_summaries.py step --
    # relabels to one arm, same trick as reg_split above.
    "../VCUTS_backup/output/0713/exp_ml_per_scene/summary_all.csv":                    "reg_per_scene",
}

parts = []
for csv_path, canon_arm in ARM_CSVS.items():
    p = Path(csv_path)
    if not p.exists():
        raise FileNotFoundError(f"{p}  (cwd={Path.cwd()})")
    sub = pd.read_csv(p)
    sub = sub[(sub["grid_shape"] == "[8, 8, 8]") & (sub["packing_mode"] == "tile_partial")
              & (sub["weight_mode"] == "screen_area") & (sub["scheme"] == "ml")].copy()
    sub["model_arm"] = canon_arm
    parts.append(sub)

# reference lines: baseline (vd_lod, no ML) + oracle_loo (LOO-MSE upper bound), same
# grid8/tile_partial/screen_area slice, sourced from the comprehensive Exp4 CSV (all 10 scenes)
_ref_csv = pd.read_csv("../VCUTS_backup/output/0704/quality_sweep/summary_all_ml_fixed.csv")  # archived 2026-07-17
_ref_csv = _ref_csv[(_ref_csv["grid_shape"] == "[8, 8, 8]") & (_ref_csv["packing_mode"] == "tile_partial")
                     & (_ref_csv["weight_mode"] == "screen_area")
                     & (_ref_csv["scheme"].isin(["vd_lod", "oracle_loo"]))].copy()
_ref_csv["model_arm"] = _ref_csv["scheme"]  # "vd_lod" / "oracle_loo" become their own arms
parts.append(_ref_csv)

df_all = pd.concat(parts, ignore_index=True)

# provenance: exact sliced rows behind this comparison's figures, alongside the PNGs
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df_all.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

df_real3  = df_all[df_all["scene"].isin(REAL3_SCENES)].copy()
df_synth7 = df_all[df_all["scene"].isin(SYNTH7_SCENES)].copy()

rows_all10  = _apply_budget_labels(df_all.to_dict("records"), BUDGET_PCTS)
rows_real3  = _apply_budget_labels(df_real3.to_dict("records"), BUDGET_PCTS)
rows_synth7 = _apply_budget_labels(df_synth7.to_dict("records"), BUDGET_PCTS)

print(f"all:    {len(rows_all10)} rows | arms: {sorted(set(r['model_arm'] for r in rows_all10))}")
print(f"real3:  {len(rows_real3)} rows | arms: {sorted(set(r['model_arm'] for r in rows_real3))}")
print(f"synth7: {len(rows_synth7)} rows | arms: {sorted(set(r['model_arm'] for r in rows_synth7))}")

all:    60000 rows | arms: ['oracle_loo', 'reg_all', 'reg_per_scene', 'reg_split', 'vd_lod']
real3:  18000 rows | arms: ['oracle_loo', 'reg_all', 'reg_per_scene', 'reg_split', 'vd_lod']
synth7: 42000 rows | arms: ['oracle_loo', 'reg_all', 'reg_per_scene', 'reg_split', 'vd_lod']


In [33]:
# rank arms removed 2026-07-13 (formally rejected -- see rvpmd01 above)
ARM_KEY_CONFIG = {
    "oracle_loo":    {"label": "Oracle (upper bound)",           "marker": "*", "color": "black",         "ls": ":"},
    "reg_all":       {"label": "Regressor (all-pool, deployed)", "marker": "s", "color": color_palette[0], "ls": "-"},
    "reg_split":     {"label": "Regressor (domain-split)",       "marker": "^", "color": color_palette[2], "ls": "-"},
    "reg_per_scene": {"label": "Regressor (per-scene)",          "marker": "v", "color": color_palette[6], "ls": "-"},
    "vd_lod":        {"label": "Heuristic (baseline, no ML)",    "marker": "D", "color": "#555555",        "ls": "--"},
}
ARM_ORDER = ["oracle_loo", "reg_all", "reg_split", "reg_per_scene", "vd_lod"]
CDF_BUDGET_PCT = 40  # standard reference budget used throughout this project's candidate comparisons

In [34]:
agg_all    = _aggregate(rows_all10,  GROUP_BY)
agg_real3  = _aggregate(rows_real3,  GROUP_BY)
agg_synth7 = _aggregate(rows_synth7, GROUP_BY)

order_all    = [k for k in ARM_ORDER if k in agg_all]
order_real3  = [k for k in ARM_ORDER if k in agg_real3]
order_synth7 = [k for k in ARM_ORDER if k in agg_synth7]

from collections import defaultdict as _dd
_by_scene_real3  = _dd(list)
_by_scene_synth7 = _dd(list)
for r in rows_real3:
    _by_scene_real3[r["scene"]].append(r)
for r in rows_synth7:
    _by_scene_synth7[r["scene"]].append(r)
scenes_agg_real3  = {s: _aggregate(_by_scene_real3[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene_real3}
scenes_agg_synth7 = {s: _aggregate(_by_scene_synth7[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene_synth7}

print(f"all order:    {order_all}")
print(f"real3 order:  {order_real3}")
print(f"synth7 order: {order_synth7}")

all order:    ['oracle_loo', 'reg_all', 'reg_split', 'reg_per_scene', 'vd_lod']
real3 order:  ['oracle_loo', 'reg_all', 'reg_split', 'reg_per_scene', 'vd_lod']
synth7 order: ['oracle_loo', 'reg_all', 'reg_split', 'reg_per_scene', 'vd_lod']


In [35]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

def plot_metric_3panel(metric, ylabel, out_stem):
    fig, axes = plt.subplots(1, 3, figsize=(figsize[0] * 2.8, figsize[1] * 1.05), sharey=True)
    all_means = []
    for ax, title, agg, order in zip(axes,
        ["all (10 scenes)", "real3", "synth7"],
        [agg_all, agg_real3, agg_synth7],
        [order_all, order_real3, order_synth7]):
        for key in order:
            cfg = ARM_KEY_CONFIG[key]
            bks = sorted(agg[key], key=_bk_sort)
            xs  = [_bk_sort(bk) for bk in bks]
            ys  = [agg[key][bk][f"{metric}_mean"] for bk in bks]
            es  = [agg[key][bk][f"{metric}_ci95"] for bk in bks]
            all_means.extend(ys)
            ax.errorbar(xs, ys, yerr=es, marker=cfg["marker"], color=cfg["color"], linestyle=cfg["ls"],
                        linewidth=2.3, markersize=7, capsize=err_capsize, elinewidth=err_lw,
                        capthick=err_capthick, label=cfg["label"], zorder=2)
        ax.set_title(title, fontsize=17)
        ax.set_xlabel(r"Budget (\% of Scene)", fontsize=15)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}\%"))
        if metric == "psnr":
            ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.0, zorder=0, alpha=0.5)
        ax.tick_params(labelsize=13)
        for spine in ax.spines.values():
            spine.set_visible(True)
        ax.tick_params(direction="out", which="both", top=False, right=False)
    axes[0].set_ylabel(ylabel, fontsize=17)
    axes[0].set_ylim(*_data_ylim(all_means, metric))
    handles = [plt.Line2D([0], [0], color=ARM_KEY_CONFIG[k]["color"], linestyle=ARM_KEY_CONFIG[k]["ls"],
                           marker=ARM_KEY_CONFIG[k]["marker"], linewidth=2.3) for k in ARM_ORDER]
    labels = [ARM_KEY_CONFIG[k]["label"] for k in ARM_ORDER]
    fig.legend(handles, labels, loc="upper center", ncol=len(labels), fontsize=13,
               framealpha=0.9, bbox_to_anchor=(0.5, 1.14))
    fig.set_constrained_layout(True)

    out_path = out_dir / f"{out_stem}.png"
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

plot_metric_3panel("psnr",  r"Quality in PSNR (dB)",    "psnr_vs_budget")
plot_metric_3panel("ssim",  r"Quality in SSIM",         "ssim_vs_budget")
plot_metric_3panel("lpips", r"LPIPS (Lower Is Better)", "lpips_vs_budget")

# per-scene grids (real3/synth7 only -- an "all" grid would just be their union, redundant)
for metric, ylabel in [("psnr", r"Quality in PSNR (dB)"), ("ssim", r"Quality in SSIM"), ("lpips", r"LPIPS (Lower Is Better)")]:
    plot_grid(scenes_agg_real3,  order_real3,  metric, ylabel, grid_dir / f"real3_{metric}_vs_budget.png",  ncols=3, key_config=ARM_KEY_CONFIG)
    plot_grid(scenes_agg_synth7, order_synth7, metric, ylabel, grid_dir / f"synth7_{metric}_vs_budget.png", ncols=4, key_config=ARM_KEY_CONFIG)

# clean up superseded 2-domain-panel top-level files from the old (pre-3-panel, 8-arm) design
for stale in ["real3_psnr_vs_budget", "real3_ssim_vs_budget", "real3_lpips_vs_budget",
              "synth7_psnr_vs_budget", "synth7_ssim_vs_budget", "synth7_lpips_vs_budget"]:
    for ext in [".png", ".eps"]:
        p = out_dir / f"{stale}{ext}"
        if p.exists():
            p.unlink()

print("Done.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/rank_vs_pool_comparison/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/rank_vs_pool_comparison/ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote output/quickplots/rank_vs_pool_comparison/_grid/synth7_ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/rank_vs_pool_comparison/_grid/real3_lpips_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/rank_vs_pool_comparison/_grid/synth7_lpips_vs_budget.png
Done.


### CDF, one figure / 3 panels (all / real3 / synth7) — regressor vs ranker, all-pool vs domain-split, at 40% budget

Same 6 arms as the budget-line plots above, same relabeling (`reg_split`/`rank_split` resolve
to whichever domain-matched pooled model applies per scene -- see markdown at the top of this
section). PSNR clipped at 60dB (project saturation convention) at the 40%-budget reference point.

**Read direction:** lower / further-right at a given PSNR = better. Oracle should sit lowest.

In [36]:
rows_all10_40  = [r for r in rows_all10  if r["_budget_key"] == f"{CDF_BUDGET_PCT}%"]
rows_real3_40  = [r for r in rows_real3  if r["_budget_key"] == f"{CDF_BUDGET_PCT}%"]
rows_synth7_40 = [r for r in rows_synth7 if r["_budget_key"] == f"{CDF_BUDGET_PCT}%"]

def _cdf_dict(rows):
    d = {k: [r["psnr"] for r in rows if r["model_arm"] == k] for k in ARM_ORDER}
    return {k: np.sort(np.minimum(np.asarray(v, dtype=float), PSNR_SATURATION_DB)) for k, v in d.items()}

CDF_ALL    = _cdf_dict(rows_all10_40)
CDF_REAL3  = _cdf_dict(rows_real3_40)
CDF_SYNTH7 = _cdf_dict(rows_synth7_40)
for name, d in [("all", CDF_ALL), ("real3", CDF_REAL3), ("synth7", CDF_SYNTH7)]:
    print(name, ": " + ", ".join(f"{k} n={len(v)}" for k, v in d.items()))

all : oracle_loo n=1500, reg_all n=1500, reg_split n=1500, reg_per_scene n=1500, vd_lod n=1500
real3 : oracle_loo n=450, reg_all n=450, reg_split n=450, reg_per_scene n=450, vd_lod n=450
synth7 : oracle_loo n=1050, reg_all n=1050, reg_split n=1050, reg_per_scene n=1050, vd_lod n=1050


In [37]:
fig, axes = plt.subplots(1, 3, figsize=(figsize[0] * 2.8, figsize[1] * 1.05), sharey=True)
for ax, title, cdf_dict in zip(axes,
    ["all (10 scenes)", "real3", "synth7"],
    [CDF_ALL, CDF_REAL3, CDF_SYNTH7]):
    for key in ARM_ORDER:
        vals = cdf_dict[key]
        if len(vals) == 0:
            continue
        cfg = ARM_KEY_CONFIG[key]
        y = np.arange(1, len(vals) + 1) / len(vals)
        ax.plot(vals, y, color=cfg["color"], linestyle=cfg["ls"], linewidth=2.3,
                label=cfg["label"], zorder=2)
    ax.set_title(title, fontsize=17)
    ax.set_xlabel(f"PSNR (dB) @ {CDF_BUDGET_PCT}\% Budget", fontsize=15)
    ax.tick_params(labelsize=13)
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)
axes[0].set_ylabel("Cumulative Fraction", fontsize=17)
axes[0].set_ylim(0, 1)
handles = [plt.Line2D([0], [0], color=ARM_KEY_CONFIG[k]["color"], linestyle=ARM_KEY_CONFIG[k]["ls"], linewidth=2.3)
           for k in ARM_ORDER]
labels = [ARM_KEY_CONFIG[k]["label"] for k in ARM_ORDER]
fig.legend(handles, labels, loc="upper center", ncol=len(labels), fontsize=13,
           framealpha=0.9, bbox_to_anchor=(0.5, 1.14))
fig.set_constrained_layout(True)

out_path = out_dir / "cdf.png"
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
print(f"Wrote {out_path}")
plt.close(fig)
print("Done.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/rank_vs_pool_comparison/cdf.png
Done.


## Exp1 scope comparison (2026-07-13, quick-inspect, NOT paper-final)

`--gs-weight-scope full` vs `visible` (culled GS-weight compute). `full` = existing canonical
grid8/progressive/screen_area/vd_lod slice; `visible` = new run,
`../VCUTS_backup/output/0713/exp1_culled_weight/` (archived 2026-07-17). Combined per-scene CSVs (tagged with a synthesized `scope`
column) built and saved alongside this section's output. Output to
`output/quickplots/` — this comparison isn't paper-decided, per PLAN.md.

In [38]:
SUMMARY_CSV = "../VCUTS_backup/output/0713/exp1_culled_weight/_combined_scope/summary_all.csv"
OUT_DIR     = "output/quickplots/0713_exp1_scope_comparison/"
GROUP_BY    = "scope"
BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")


Loaded 24000 rows | groups: ['full', 'visible'] | budgets: ['10%', '25%', '40%', '55%', '70%', '85%', '99%', '100%']


In [39]:
SCOPE_KEY_CONFIG = {
    "full":    {"label": "full scope",    "marker": "s", "color": color_palette[0]},
    "visible": {"label": "visible scope", "marker": "^", "color": color_palette[3]},
}
SCOPE_ORDER = ["full", "visible"]

agg = _aggregate(rows, GROUP_BY)
order = [k for k in SCOPE_ORDER if k in agg]
labels = {k: SCOPE_KEY_CONFIG[k]["label"] for k in order}

_by_scene = _dd(list)
for r in rows:
    _by_scene[r["scene"]].append(r)
scenes_agg = {s: _aggregate(_by_scene[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene}

print(f"order: {order}")
print(f"scenes: {list(scenes_agg.keys())}")


order: ['full', 'visible']
scenes: ['bicycle', 'garden', 'stump', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship']


In [40]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir, key_config=SCOPE_KEY_CONFIG)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir, key_config=SCOPE_KEY_CONFIG)
plot_metric(agg, order, "lpips", r"LPIPS (Lower Is Better)", "lpips_vs_budget", out_dir, key_config=SCOPE_KEY_CONFIG)

plot_grid(scenes_agg, order, "psnr", r"Quality in PSNR (dB)", grid_dir / "psnr_vs_budget.png", ncols=5, key_config=SCOPE_KEY_CONFIG)
plot_grid(scenes_agg, order, "ssim", r"Quality in SSIM", grid_dir / "ssim_vs_budget.png", ncols=5, key_config=SCOPE_KEY_CONFIG)
plot_grid(scenes_agg, order, "lpips", r"LPIPS (Lower Is Better)", grid_dir / "lpips_vs_budget.png", ncols=5, key_config=SCOPE_KEY_CONFIG)
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0713_exp1_scope_comparison/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0713_exp1_scope_comparison/ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote output/quickplots/0713_exp1_scope_comparison/_grid/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0713_exp1_scope_comparison/_grid/ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0713_exp1_scope_comparison/_grid/lpips_vs_budget.png
Done.


## Exp2 heuristics-only formula comparison (2026-07-13, quick-inspect, NOT paper-final)

4 heuristic schemes, NO ml: `vd_lod` (baseline), `v_lod_w` (canonical/ours), `vd_lod_w` (old
rejected, fresh rerun under current pipeline), `w_lod` (new, `W_k` alone, needs culled
`--gs-weight-scope visible`). Combined per-scene CSVs built and saved alongside this
section's output. Output to `output/quickplots/` — not paper-decided.

In [41]:
SUMMARY_CSV = "../VCUTS_backup/output/0713/exp2_formula_comparison/_combined_heuristics/summary_all.csv"
OUT_DIR     = "output/quickplots/0713_exp2_heuristics_comparison/"
GROUP_BY    = "scheme"
BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")


Loaded 48000 rows | groups: ['v_lod_w', 'vd_lod', 'vd_lod_w', 'w_lod'] | budgets: ['10%', '25%', '40%', '55%', '70%', '85%', '99%', '100%']


In [42]:
HEUR_ORDER = ["vd_lod", "v_lod_w", "vd_lod_w", "w_lod"]
HEUR_KEY_CONFIG = dict(KEY_CONFIG)
HEUR_KEY_CONFIG["w_lod"] = {"label": "w_lod (W_k alone, new)", "marker": "D", "color": color_palette[8]}

agg = _aggregate(rows, GROUP_BY)
order = [k for k in HEUR_ORDER if k in agg]
labels = {k: HEUR_KEY_CONFIG[k]["label"] for k in order}

_by_scene = _dd(list)
for r in rows:
    _by_scene[r["scene"]].append(r)
scenes_agg = {s: _aggregate(_by_scene[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene}

print(f"order: {order}")
print(f"scenes: {list(scenes_agg.keys())}")


order: ['vd_lod', 'v_lod_w', 'vd_lod_w', 'w_lod']
scenes: ['bicycle', 'garden', 'stump', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship']


In [43]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir, key_config=HEUR_KEY_CONFIG)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir, key_config=HEUR_KEY_CONFIG)
plot_metric(agg, order, "lpips", r"LPIPS (Lower Is Better)", "lpips_vs_budget", out_dir, key_config=HEUR_KEY_CONFIG)

plot_grid(scenes_agg, order, "psnr", r"Quality in PSNR (dB)", grid_dir / "psnr_vs_budget.png", ncols=5, key_config=HEUR_KEY_CONFIG)
plot_grid(scenes_agg, order, "ssim", r"Quality in SSIM", grid_dir / "ssim_vs_budget.png", ncols=5, key_config=HEUR_KEY_CONFIG)
plot_grid(scenes_agg, order, "lpips", r"LPIPS (Lower Is Better)", grid_dir / "lpips_vs_budget.png", ncols=5, key_config=HEUR_KEY_CONFIG)
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0713_exp2_heuristics_comparison/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0713_exp2_heuristics_comparison/ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote output/quickplots/0713_exp2_heuristics_comparison/_grid/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0713_exp2_heuristics_comparison/_grid/ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0713_exp2_heuristics_comparison/_grid/lpips_vs_budget.png
Done.
